In [ ]:
import os
import json
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
# ====== ตั้งค่า ======
CSV_PATH = "jobs_search_master.csv"
COLLECTION_NAME = "job_search_vector"
PERSIST_DIR = "./chroma_db"
USE_EMBEDDING = True
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
BATCH_SIZE = 1000

In [ ]:
# ====== ฟังก์ชันช่วย ======
def safe_metadata(v):
    if v is None or pd.isna(v):
        return ""
    if isinstance(v, (str, int, float, bool)):
        return v
    try:
        return json.dumps(v, ensure_ascii=False)
    except:
        return str(v)

def make_doc(row, cols):
    parts = [str(row[c]) for c in cols if c in row and row[c]]
    return "\n".join(parts).strip()

In [ ]:
def ingest_data():
    """ฟังก์ชันสำหรับ ingest ข้อมูลเข้า ChromaDB"""
    # โหลด CSV
    df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False,
                     na_values=["", "NULL", "nan", "NaN", "0000-00-00"])
    print(f"โหลดข้อมูล {len(df)} แถว")

    # ==== เตรียม documents, ids, metadatas ====
    if "id" in df.columns:
        if df["id"].duplicated().any():
            df["__id"] = df["id"].astype(str) + "_" + df.index.astype(str)
        else:
            df["__id"] = df["id"].astype(str)
    else:
        df["__id"] = df.index.astype(str)

    doc_cols = ["position", "company_name", "company_name_eng", "detail"]
    documents, ids, metadatas = [], [], []

    for _, row in df.iterrows():
        doc = make_doc(row, doc_cols)
        if not doc:
            doc = str(row.get("position", "")) or str(row.get("job_pos", ""))
        documents.append(doc)
        ids.append(row["__id"])

        md = {col: safe_metadata(row[col])
              for col in ["jobpost_id", "company_id", "position", "company_name", "public_date", "end_date"]
              if col in row}
        metadatas.append(md)

    print(f"ตัวอย่าง document:\n{documents[0][:200]}")

    # ==== Connect Chroma ====
    client = chromadb.Client(
        Settings(
            persist_directory=PERSIST_DIR,
            anonymized_telemetry=False
        )
    )

    try:
        collection = client.get_collection(COLLECTION_NAME)
        print(f"ใช้ collection เดิม: {COLLECTION_NAME}")
    except:
        collection = client.create_collection(COLLECTION_NAME)
        print(f"สร้าง collection ใหม่: {COLLECTION_NAME}")

    # ==== Embeddings ====
    model = None
    embeddings = None
    if USE_EMBEDDING:
        model = SentenceTransformer(EMBED_MODEL)
        print("กำลังสร้าง embeddings ...")
        embeddings = model.encode(documents, show_progress_bar=True).tolist()

    # ==== เพิ่มข้อมูลเข้า collection ====
    for i in range(0, len(ids), BATCH_SIZE):
        batch_ids = ids[i:i+BATCH_SIZE]
        batch_docs = documents[i:i+BATCH_SIZE]
        batch_mds = metadatas[i:i+BATCH_SIZE]
        batch_emb = embeddings[i:i+BATCH_SIZE] if embeddings else None

        if batch_emb:
            collection.add(ids=batch_ids, documents=batch_docs, metadatas=batch_mds, embeddings=batch_emb)
        else:
            collection.add(ids=batch_ids, documents=batch_docs, metadatas=batch_mds)

    print("✅ เพิ่มข้อมูลเข้า Local ChromaDB เสร็จแล้ว")

In [ ]:
def query_jobs(queries, top_k=5):
    """ฟังก์ชันสำหรับค้นหางานตาม queries"""
    
    # โหลด model สำหรับ encode query
    model = SentenceTransformer(EMBED_MODEL)
    
    # เชื่อมต่อ ChromaDB
    client = chromadb.Client(
        Settings(
            persist_directory=PERSIST_DIR,
            anonymized_telemetry=False
        )
    )
    
    try:
        collection = client.get_collection(COLLECTION_NAME)
        print(f"✅ เชื่อมต่อ collection: {COLLECTION_NAME}\n")
    except Exception as e:
        print(f"❌ ไม่พบ collection: {e}")
        return
    
    # วนลูปแต่ละ query
    for i, query in enumerate(queries, 1):
        print(f"\n{'='*80}")
        print(f"🔍 Query {i}: {query}")
        print(f"{'='*80}")
        
        # Encode query
        query_embedding = model.encode([query]).tolist()
        
        # ค้นหา
        results = collection.query(
            query_embeddings=query_embedding,
            n_results=top_k
        )
        
        # แสดงผล
        if results['documents'] and results['documents'][0]:
            for j, (doc, metadata, distance) in enumerate(
                zip(results['documents'][0], 
                    results['metadatas'][0], 
                    results['distances'][0]), 1
            ):
                print(f"\n📋 อันดับ {j} (คะแนน: {distance:.4f})")
                print(f"   ตำแหน่ง: {metadata.get('position', 'N/A')}")
                print(f"   บริษัท: {metadata.get('company_name', 'N/A')}")
                print(f"   วันประกาศ: {metadata.get('public_date', 'N/A')}")
                print(f"   สิ้นสุดวันที่: {metadata.get('end_date', 'N/A')}")
                print(f"   เอกสาร: {doc[:150]}..." if len(doc) > 150 else f"   เอกสาร: {doc}")
        else:
            print("   ❌ ไม่พบผลลัพธ์")
    
    print(f"\n{'='*80}")
    print("✅ เสร็จสิ้นการค้นหา")


if __name__ == "__main__":
    # เลือกโหมดการทำงาน
    import sys
    
    if len(sys.argv) > 1 and sys.argv[1] == "ingest":
        # 🚀 Run ingestion
        print("🚀 เริ่มต้น Ingestion...")
        ingest_data()
    else:
        # 🔍 Run queries
        print("🔍 เริ่มต้น Query...")
        queries = [
            # --- responsibilities ---
            "อยากหางาน Developer",
            "หางานที่ต้องจัดทำรายงานการขายประจำเดือน",
            "มองหางานที่ดูแลระบบเครือข่ายและ Server",
        ]
        
        query_jobs(queries, top_k=10)

In [ ]:
queries = [
    "กำลังหางาน Web Application Developer ด้วย Django ทำงาน 5 วัน จันทร์-ศุกร์ เงินเดือน 20000 - 30000 บาท ในสายไอที / ซอฟต์แวร์",
    "มองหางานที่ต้องจัดทำรายงานการขายประจำเดือน เป็น Marketing Executive งานประจำ Full-time เงินเดือน 35000 - 45000 บาท ในสายธนาคาร / การเงิน",
    "สนใจงานที่ดูแลระบบเครือข่ายและ Server ตำแหน่ง Data Analyst งานพิเศษ Part-time เงินเดือน 50000 - 70000 บาท ในสายการแพทย์ / สาธารณสุข",
    "อยากได้งาน Backend Developer ที่ต้องออกแบบและพัฒนาระบบ Web Application ด้วย Django งานประจำ Full-time เงินเดือน 35000 - 45000 บาท ในสายไอที / ซอฟต์แวร์",
    "หางาน Marketing Executive ที่ต้องจัดทำรายงานการขายประจำเดือน ทำงาน 5 วัน จันทร์-ศุกร์ เงินเดือน 20000 - 30000 บาท ในสายธนาคาร / การเงิน",
    "กำลังมองหาตำแหน่ง Data Analyst ดูแลระบบเครือข่ายและ Server งานพิเศษ Part-time เงินเดือน 50000 - 70000 บาท ในสายการแพทย์ / สาธารณสุข"
]

query_jobs(queries, top_k=5)